# SignVision AI — Export CNN for Vercel

This notebook does **not** retrain your model. It removes the training-only augmentation submodel, preserves scaling and learned layers, and exports an HDF5 file that can be converted to TensorFlow.js.

Run this in Colab using the same Google Drive model from your training notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import tensorflow as tf

model_path = '/content/drive/MyDrive/SignLanguage/TrainedModel/hand_sign_cnn.keras'
labels_path = '/content/drive/MyDrive/SignLanguage/TrainedModel/class_names.json'
trained_model = tf.keras.models.load_model(model_path, compile=False)
with open(labels_path, 'r', encoding='utf-8') as f:
    class_names = json.load(f)
print('Input:', trained_model.input_shape)
print('Output:', trained_model.output_shape)
print('Labels:', class_names)
print('Layers:', [layer.name for layer in trained_model.layers])

In [ ]:
rescale_index = next(i for i, layer in enumerate(trained_model.layers) if isinstance(layer, tf.keras.layers.Rescaling))
inputs = tf.keras.Input(shape=(128, 128, 3), name='image')
x = inputs
for layer in trained_model.layers[rescale_index:]:
    x = layer(x)
inference_model = tf.keras.Model(inputs, x, name='signvision_inference')
inference_model.save('/content/signvision_inference.h5', include_optimizer=False)
with open('/content/class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)
print('Saved /content/signvision_inference.h5')
print('Output shape:', inference_model.output_shape)
print('Next: download signvision_inference.h5 and class_names.json, then follow MODEL_EXPORT_GUIDE.md to convert the H5 file with tensorflowjs_converter.')